In [34]:
print("=" * 70)
print("              GROUPDNA REPORT")
print("=" * 70)

with open("/content/DADS Minor PROJECT dataset.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

messages = []

for line in lines:
    line = line.strip()

    if line == "":
        continue

    parts = line.split(" - ", 1)

    if len(parts) < 2:
        continue

    info = parts[1].split(": ", 1)

    if len(info) < 2:
        continue

    message = {
        "timestamp": parts[0],
        "sender": info[0],
        "text": info[1]
    }

    messages.append(message)

# print(messages[:5])
# print(len(messages))
participants = set()

for msg in messages:
    participants.add(msg["sender"])

# print(participants)
# print(f"No. of senders: {len(participants)} ")

person_count = {}
for msg in messages:
    sender = msg["sender"]

    if sender in person_count:
        person_count[sender] += 1
    else:
        person_count[sender] = 1
# print(person_count)

from datetime import datetime
dates = []

for msg in messages:
    date = msg["timestamp"].split(",")[0]
    dates.append(date)

first_date = datetime.strptime(dates[0], "%d/%m/%y")

last_date = datetime.strptime(dates[-1], "%d/%m/%y")
total_days = (last_date - first_date).days + 1

print(f"Period        : {dates[0]} to {dates[-1]}")
print(f"Total Days    : {total_days}")
print(f"Participants  : {len(participants)}")
print(f"Messages      : {len(messages)}")

print("=" * 70)

# Message Count
print("\n1. MESSAGES PER PERSON\n")

sorted_people = sorted(
    person_count.items(),
    key=lambda x:x[1],
    reverse=True
)
# print(sorted_people)

for person,count in sorted_people:
    percentage = (count / len(messages))*100
    bar = "█" * int(percentage)
    print(f"{person:<8} {bar:<30} {count} ({percentage:.1f}%)")

print("=" * 70)

# Day 3
word_count = {}

stop_words = {
    "i","is","the","a","an","and","or","to","of","in","on","for","it","this","that","are","was","am","be","with","you","we","my","me","he","his","she","they","about",
    "how","at","so","have","has","had","just","from","by","our","your","their","there","here","today","now","hai","started",
    "telling","which","everyone","anyone","up","one","no","entire","but","everything","came","please","who","what","cant","why","been","sleep","three","used"
}

for msg in messages:

    text = msg["text"]

    # Skip media messages
    if text == "<Media omitted>":
        continue

    # Skip deleted messages
    if text == "This message was deleted":
        continue

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.replace(",", "")
    text = text.replace(".", "")
    text = text.replace("!", "")
    text = text.replace("?", "")
    text = text.replace(":", "")
    text = text.replace(";", "")
    text = text.replace("(", "")
    text = text.replace(")", "")
    text = text.replace('"', "")
    text = text.replace("'", "")

    # Split sentence into words
    words = text.split()

    # Count words
    for word in words:

        # Skip stop words
        if word in stop_words:
            continue

        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

# print(word_count)

# Sort words
sorted_words = sorted(
    word_count.items(),
    key=lambda x: x[1],
    reverse=True
)

# print(sorted_words[:10])
# Print Top 10 with bars

print("\n2. Most Frequent Words from group chat: \n")
for word, count in sorted_words[:10]:
    bar = "█" * (count // 10)
    print(f"{word:<12} {bar} {count}")

print("=" * 70)

# Day 4

import numpy as np

participants = sorted(list(set(msg["sender"] for msg in messages)))

person_index = {}

for i, person in enumerate(participants):
    person_index[person] = i

heatmap = np.zeros((len(participants), 24), dtype=int)

for msg in messages:

    sender = msg["sender"]
    row = person_index[sender]

    time = msg["timestamp"].split(",")[1].strip()
    hour = int(time.split(":")[0])

    heatmap[row][hour] += 1

# print(heatmap)
# print("Hour → 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23")

print("\n3. ACTIVITY HEATMAP (Messages by Hour)\n")

# Print selected hours
print("       00 03 06 09 12 15 18 21")

hours = [0, 3, 6, 9, 12, 15, 18, 21]

for i, person in enumerate(participants):

    print(f"{person:<7}", end=" ")

    max_value = max(heatmap[i])

    for hour in hours:

        value = heatmap[i][hour]

        if value == 0:
            symbol = "."

        else:
            ratio = value / max_value

            if ratio <=     0.25:
                symbol = "░"

            elif ratio <=   0.50:
                symbol = "▒"
            elif ratio <=   0.75:
                symbol = "▒"

            else:
                symbol = "█"

        print(symbol, end="  ")

    print()

# Day 5 : Response Patterns

from datetime import datetime, timedelta

# Convert timestamp to datetime
for msg in messages:
    msg["datetime"] = datetime.strptime(
        msg["timestamp"],
        "%d/%m/%y, %H:%M"
    )

# --------------------------
# Part A : Response Time
# --------------------------

response_times = {}

for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]

    # Ignore consecutive messages by same sender
    if previous["sender"] == current["sender"]:
        continue

    gap = current["datetime"] - previous["datetime"]
    minutes = gap.total_seconds() / 60

    sender = current["sender"]

    if sender not in response_times:
        response_times[sender] = []

    response_times[sender].append(minutes)

# Average response time
average_response = {}

for person in response_times:

    total = 0

    for minute in response_times[person]:
        total += minute

    average = total / len(response_times[person])

    average_response[person] = average

print("=" * 50)
print("RESPONSE PATTERNS")
print("=" * 50)

for person in average_response:
    print(f"{person:<8}: {average_response[person]:.2f} minutes")

# --------------------------
# Fastest Replier
# --------------------------

fastest = ""
fastest_time = float("inf")

for person in average_response:

    if average_response[person] < fastest_time:

        fastest_time = average_response[person]
        fastest = person

# --------------------------
# Slowest Replier
# --------------------------

slowest = ""
slowest_time = 0

for person in average_response:

    if average_response[person] > slowest_time:

        slowest_time = average_response[person]
        slowest = person

print()
print(f"Fastest Replier : {fastest} ({fastest_time:.2f} minutes)")
print(f"Slowest Replier : {slowest} ({slowest_time:.2f} minutes)")


# Part B : Longest Silent Streak

# Store active dates
active_dates = {}

for person in participants:
    active_dates[person] = set()

for msg in messages:

    sender = msg["sender"]

    date = msg["timestamp"].split(",")[0]

    active_dates[sender].add(date)

# Create complete date list
all_dates = []

current_date = first_date

while current_date <= last_date:

    all_dates.append(
        current_date.strftime("%d/%m/%y")
    )

    current_date += timedelta(days=1)

# Find longest silent streak
silent_streak = {}

for person in participants:

    current_streak = 0
    longest_streak = 0

    for date in all_dates:

        if date not in active_dates[person]:

            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak

        else:
            current_streak = 0

    silent_streak[person] = longest_streak

print()
print("=" * 50)
print("LONGEST SILENT STREAKS")
print("=" * 50)

for person in silent_streak:
    print(f"{person:<8}: {silent_streak[person]} days")


print("=" * 50)
print("SPAM SCORE")
print("=" * 50)

burst_data = {}

current_sender = messages[0]["sender"]
current_burst = 1

for i in range(1, len(messages)):

    if messages[i]["sender"] == current_sender:
        current_burst += 1

    else:

        if current_sender not in burst_data:
            burst_data[current_sender] = []

        burst_data[current_sender].append(current_burst)

        current_sender = messages[i]["sender"]
        current_burst = 1

# Store last burst
if current_sender not in burst_data:
    burst_data[current_sender] = []

burst_data[current_sender].append(current_burst)

spam_score = {}

for person in burst_data:

    total = 0

    for burst in burst_data[person]:
        total += burst

    spam_score[person] = total / len(burst_data[person])

for person in spam_score:
    print(f"{person:<8}: {spam_score[person]:.2f}")



print("\n" + "=" * 50)
print("CARING FRIEND SCORE")
print("=" * 50)

care_words = {
    "okay","safe","eat","sleep","take",
    "care","please","drink","water",
    "forget","reminder"
}

mom_score = {}

for person in participants:
    mom_score[person] = 0

for msg in messages:

    sender = msg["sender"]

    words = msg["text"].lower().split()

    for word in words:

        if word in care_words:
            mom_score[sender] += 1

for person in mom_score:
    print(f"{person:<8}: {mom_score[person]}")


print("\n" + "=" * 50)
print("NIGHT OWL SCORE")
print("=" * 50)

night_score = {}

for person in participants:

    total = 0
    night = 0

    for msg in messages:

        if msg["sender"] == person:

            total += 1

            hour = int(msg["timestamp"].split(",")[1].split(":")[0])

            if hour >= 23 or hour <= 4:
                night += 1

    percentage = (night / total) * 100

    night_score[person] = percentage

for person in night_score:
    print(f"{person:<8}: {night_score[person]:.2f}%")

print("\n" + "=" * 50)
print("STORY TELLER SCORE")
print("=" * 50)

story_score = {}

for person in participants:

    total_words = 0
    total_messages = 0

    for msg in messages:

        if msg["sender"] == person:

            total_messages += 1

            total_words += len(msg["text"].split())

    story_score[person] = total_words / total_messages

for person in story_score:
    print(f"{person:<8}: {story_score[person]:.2f} words/message")

print("\n" + "=" * 50)
print("DRAMA SCORE")
print("=" * 50)

drama_score = {}

for person in participants:

    total = 0
    drama = 0

    for msg in messages:

        if msg["sender"] == person:

            total += 1

            text = msg["text"]

            if text.isupper() or text.count("!") >= 2:
                drama += 1

    percentage = (drama / total) * 100

    drama_score[person] = percentage

for person in drama_score:
    print(f"{person:<8}: {drama_score[person]:.2f}%")


print("\n" + "=" * 50)
print("COMEDY SCORE")
print("=" * 50)

fun_words = {
    "lol","haha","lmao","rofl","lmfao"
}

comedy_score = {}

for person in participants:
    comedy_score[person] = 0

for msg in messages:

    sender = msg["sender"]

    words = msg["text"].lower().split()

    for word in words:

        if word in fun_words:
            comedy_score[sender] += 1

for person in comedy_score:
    print(f"{person:<8}: {comedy_score[person]}")


print("\n" + "=" * 50)
print("QUESTION ASKER SCORE")
print("=" * 50)

question_score = {}

for person in participants:

    total = 0
    questions = 0

    for msg in messages:

        if msg["sender"] == person:

            total += 1

            if msg["text"].endswith("?"):
                questions += 1

    percentage = (questions / total) * 100

    question_score[person] = percentage

for person in question_score:
    print(f"{person:<8}: {question_score[person]:.2f}%")


print("\n" + "=" * 50)
print("FOODIE SCORE")
print("=" * 50)

food_words = {
    "chai","coffee","tea","biryani","maggi",
    "dosa","canteen","lunch","dinner","eat"
}

foodie_score = {}

for person in participants:
    foodie_score[person] = 0

for msg in messages:

    sender = msg["sender"]

    words = msg["text"].lower().split()

    for word in words:

        if word in food_words:
            foodie_score[sender] += 1

for person in foodie_score:
    print(f"{person:<8}: {foodie_score[person]}")

print("\n6. PERSONALITY ARCHETYPES\n")

print(f"THE SPAMMER          : {max(spam_score,key=spam_score.get)}")
print(f"THE GROUP MOM        : {max(mom_score,key=mom_score.get)}")
print(f"THE NIGHT OWL        : {max(night_score,key=night_score.get)}")
print(f"THE STORYTELLER      : {max(story_score,key=story_score.get)}")
print(f"THE DRAMA QUEEN      : {max(drama_score,key=drama_score.get)}")
print(f"THE GHOST            : {max(silent_streak,key=silent_streak.get)}")
print(f"THE COMEDIAN         : {max(comedy_score,key=comedy_score.get)}")
print(f"THE QUESTION MASTER  : {max(question_score,key=question_score.get)}")
print(f"THE FOODIE           : {max(foodie_score,key=foodie_score.get)}")

print("=" * 70)

              GROUPDNA REPORT
Period        : 01/04/24 to 30/05/24
Total Days    : 60
Participants  : 6
Messages      : 3174

1. MESSAGES PER PERSON

Rahul    ██████████████████████████████ 953 (30.0%)
Priya    ██████████████████████         718 (22.6%)
Neha     ████████████████████           635 (20.0%)
Aman     ███████████████                490 (15.4%)
Karan    ███████████                    354 (11.2%)
Vikas                                   24 (0.8%)

2. Most Frequent Words from group chat: 

guys         ███████████████████████████████ 318
bhai         ████████████████ 160
scene        ██████████████ 145
yaar         █████████████ 139
kya          █████████████ 133
ja           ██████████ 101
raha         ██████████ 101
life         █████████ 94
aman         █████████ 93
okay         █████████ 92

3. ACTIVITY HEATMAP (Messages by Hour)

       00 03 06 09 12 15 18 21
Aman    ▒  ▒  .  .  .  ░  ░  ░  
Karan   .  .  .  ▒  █  ▒  ▒  ▒  
Neha    .  .  ░  █  ▒  ░  █  ▒  
Priya   .  .  ░